# Identifikasi Kasus "Tidak Ditemukan" pada Sensus Ekonomi 2026
Notebook ini digunakan untuk mengidentifikasi prelist yang memiliki status atau indikasi "Tidak Ditemukan", baik untuk kategori **Keluarga** maupun **Usaha**.

Hasil akhir dari notebook ini menghasilkan 5 file output:
1. `tidak_ditemukan_keluarga.csv` (Data detail keluarga tidak ditemukan)
2. `tidak_ditemukan_usaha.csv` (Data detail usaha tidak ditemukan)
3. `agregat_tidak_ditemukan_keluarga.csv` (Agregat jumlah keluarga tidak ditemukan per kecamatan)
4. `agregat_tidak_ditemukan_usaha.csv` (Agregat jumlah usaha tidak ditemukan per kecamatan)
5. `agregat_tidak_ditemukan_gabungan.csv` (Agregat gabungan keluarga + usaha tidak ditemukan per kecamatan)

## 1. Membaca Data
Membaca berkas `update_data.csv` yang berisi data hasil scraping yang sudah dibersihkan.

In [1]:
import pandas as pd
import os
import re

# Definisikan path ke file data
file_path = "../../update_data.csv"

# Membaca data
print(f"Membaca data dari: {os.path.abspath(file_path)}")
df = pd.read_csv(file_path)

# Menampilkan dimensi data
print(f"Total baris data: {df.shape[0]:,}")
print(f"Total kolom data: {df.shape[1]}")

Membaca data dari: D:\Hamdani\scraper-fasih-sm\update_data.csv
Total baris data: 55,034
Total kolom data: 19


## 2. Load Mapping Kecamatan & Koseka
Membaca file referensi `data/koseka.csv` untuk memetakan kode kecamatan ke nama kecamatan yang sesuai secara akurat.

In [2]:
# Load referensi koseka
koseka_path = "../../data/koseka.csv"
koseka_df = pd.read_csv(koseka_path, sep=';')
koseka_map = dict(zip(koseka_df['kd_kec'].astype(str), koseka_df['nama_kec']))

# Ekstrak Kode SLS (16 digit pertama) dari Kode Identitas
df['kode_wilayah'] = df['Kode Identitas'].astype(str).str.extract(r'^(\d{16})')[0].fillna('')

# Ekstrak Kode Kecamatan (7 digit pertama) dan memetakan nama kecamatan
df['kd_kec'] = df['Kode Identitas'].astype(str).str.extract(r'^(\d{7})')[0].fillna('')
df['nama_kec_ref'] = df['kd_kec'].map(koseka_map)
df['nama_kec'] = df['nama_kec_ref'].fillna(df['nama_kec'])

# Kolom yang akan dimasukkan ke dalam file output detail
output_cols = [
    'Searched Email', 'Kode Identitas', 'kode_wilayah', 'Nama Keluarga/Bangunan/Usaha', 
    'Alamat Prelist', 'Nomor Urut Bangunan / IDSBR', 'NIB', 'Email', 
    'Skala Usaha / Jenis Prelist', 'Jumlah Usaha', 'Kode Pos', 'Perubahan SLS', 
    'Status', 'Mode', 'Petugas Saat Ini', 'Keterangan', 'sumber data', 
    'nama_kec', 'koseka', 'is_prioritas'
]

## 3. Identifikasi Keluarga Tidak Ditemukan
Kriteria pemfilteran untuk **Keluarga** yang tidak ditemukan:
1. Kolom `sumber data` tidak mengandung kata "kosong" (case-insensitive).
2. `Status` adalah `approved by pengawas` (mengandung kata 'approve').
3. `Jumlah Usaha` bernilai 0, '0', atau `"-"`.
4. `Skala Usaha / Jenis Prelist` mengandung kata "keluarga" (case-insensitive).
5. `Nomor Urut Bangunan / IDSBR` berisi `"-"`, `"- / 8 digit"`, atau `8 digit` IDSBR.

In [3]:
# Bersihkan data untuk filter
df['sumber data'] = df['sumber data'].fillna('')
df['Nomor Urut Bangunan / IDSBR'] = df['Nomor Urut Bangunan / IDSBR'].fillna('').astype(str).str.strip()

# Kondisi filter umum
cond_common = (
    (~df['sumber data'].str.contains('kosong', case=False)) &
    (df['Status'].str.contains('approve', case=False, na=False)) &
    (df['Jumlah Usaha'].fillna('-').isin([0, '0', '-'])) &
    (df['Nomor Urut Bangunan / IDSBR'].str.contains(r'^-$|^\d{8}$|^-\s*/\s*\d{8}$', regex=True))
)

# Filter Keluarga
cond_keluarga = cond_common & (df['Skala Usaha / Jenis Prelist'].str.contains('keluarga', case=False, na=False))
df_kel = df[cond_keluarga][output_cols]

print(f"Jumlah Keluarga Tidak Ditemukan: {len(df_kel):,}")
df_kel.head(10)

Jumlah Keluarga Tidak Ditemukan: 684


,Searched Email,Kode Identitas,kode_wilayah,Nama Keluarga/Bangunan/Usaha,Alamat Prelist,Nomor Urut Bangunan / IDSBR,NIB,Email,Skala Usaha / Jenis Prelist,Jumlah Usaha,Kode Pos,Perubahan SLS,Status,Mode,Petugas Saat Ini,Keterangan,sumber data,nama_kec,koseka,is_prioritas
49,ningsikarendehi968@gmail.com,7103062006000100 - DTSEN - 11,7103062006000100,EDWIN ALDRIN MASIPUANG / JERLY ALDA DALAKO,KAMPUNG MALISADE,- / 12610904,-,-,UMKM/Keluarga,0,95858,-,approved by pengawas,CAPI,ningsikarendehi968@gmail.comPengawas,-,DTSEN,(062) TABUKAN SELATAN TENGGARA,Lalu,Tidak
300,jeniemonok7@gmail.com,7103091004000100 - DTSEN - 66,7103091004000100,SAARTJE MARTINI MAYA WANGET /,TONA 1,- / 12736445,-,-,UMKM/Keluarga,0,-,-,approved by pengawas,CAPI,jeniemonok7@gmail.comPengawas,-,DTSEN,(091) TAHUNA TIMUR,Djon,Tidak
394,jeniemonok7@gmail.com,7103091004000100 - DTSEN - 52,7103091004000100,ANASTHASIUS KAKASIH / ADRIANA DANTE,TONA 1,- / 13563867,-,-,UMKM/Keluarga,0,-,-,approved by pengawas,CAPI,jeniemonok7@gmail.comPengawas,-,DTSEN,(091) TAHUNA TIMUR,Djon,Tidak
944,Georgemulersasiangsasiang@gmail.com,7103090014000700 - DTSEN - 134,7103090014000700,PRAJITNO /,KELURAHAN SANTIAGO,- / 13051831,-,-,UMKM/Keluarga,0,95811,-,approved by pengawas,CAPI,georgemulersasiangsasiang@gmail.comPengawas,-,DTSEN,(090) TAHUNA,Yetty,Ya
1661,Fmanangkoda12@gmail.com,7103050005000100 - UMKM - 1,7103050005000100,SANDRO LANTEMONA / SANDRO LANTEMONA,LINDONGAN 1,- / 16685385,-,-,UMKM/Keluarga,0,95855,-,approved by pengawas,CAPI,fmanangkoda12@gmail.comPengawas,-,UMKM,(050) TAMAKO,Erike,Tidak
1945,yanfwpadang84@gmail.com,7103090007000500 - DTSEN - 31,7103090007000500,FEBIYANTI AGNETTE ABRAM / FEBIYANTI AGNETTE ABRAM,JALAN BOULEVARD,- / 16316264,-,-,UMKM/Keluarga,0,95812,-,approved by pengawas,CAPI,yanfwpadang84@gmail.comPengawas,-,DTSEN,(090) TAHUNA,Yetty,Tidak
2453,sutrisnosariang90@gmail.com,7103101004000300 - DTSEN - 17,7103101004000300,MUNAWIR MADONSA / FRASILIA MUSALER,KAMPUNG BUKIDE TIMUR,- / 15653977,-,-,UMKM/Keluarga,0,95856,2. Tidak,approved by pengawas,CAPI,sutrisnosariang90@gmail.comPengawas,-,DTSEN,(101) NUSA TABUKAN,Alfi,Tidak
2530,sutrisnosariang90@gmail.com,7103101004000300 - DTSEN - 15,7103101004000300,MASRUN MADOA / ALWIAH MINGGU,KAMPUNG NAHA,- / 12421677,-,-,UMKM/Keluarga,0,95856,2. Tidak,approved by pengawas,CAPI,sutrisnosariang90@gmail.comPengawas,-,DTSEN,(101) NUSA TABUKAN,Alfi,Tidak
2708,ningsikarendehi968@gmail.com,7103061001000200 - UMKM - 2,7103061001000200,HAROL TAKALETANGENG /,"Jln.Soa, Lindongan 02",- / 14540852,-,-,UMKM/Keluarga,0,95858,-,approved by pengawas,CAPI,ningsikarendehi968@gmail.comPengawas,-,UMKM,(061) TABUKAN SELATAN TENGAH,Erwin,Tidak
3609,DerlanMalawere92@gmail.com,7103100024000300 - DTSEN - 32,7103100024000300,RIFLI ALEXANDER MACPAL / APRELIA JOKUNG,TAPUANG,- / 12725208,-,-,UMKM/Keluarga,0,95856,-,approved by pengawas,CAPI,derlanmalawere92@gmail.comPengawas,-,DTSEN,(100) TABUKAN UTARA,Hendri,Ya


## 4. Identifikasi Usaha Tidak Ditemukan
Kriteria pemfilteran untuk **Usaha** yang tidak ditemukan:
1. Kolom `sumber data` tidak mengandung kata "kosong" (case-insensitive).
2. `Status` adalah `approved by pengawas` (mengandung kata 'approve').
3. `Jumlah Usaha` bernilai 0, '0', atau `"-"`.
4. `Skala Usaha / Jenis Prelist` **selain** yang mengandung kata "keluarga" (case-insensitive).
5. `Nomor Urut Bangunan / IDSBR` berisi `"-"`, `"- / 8 digit"`, atau `8 digit` IDSBR.

In [4]:
# Filter Usaha
cond_usaha = cond_common & (~df['Skala Usaha / Jenis Prelist'].str.contains('keluarga', case=False, na=False))
df_us = df[cond_usaha][output_cols]

print(f"Jumlah Usaha Tidak Ditemukan: {len(df_us):,}")
df_us.head(10)

Jumlah Usaha Tidak Ditemukan: 961


,Searched Email,Kode Identitas,kode_wilayah,Nama Keluarga/Bangunan/Usaha,Alamat Prelist,Nomor Urut Bangunan / IDSBR,NIB,Email,Skala Usaha / Jenis Prelist,Jumlah Usaha,Kode Pos,Perubahan SLS,Status,Mode,Petugas Saat Ini,Keterangan,sumber data,nama_kec,koseka,is_prioritas
274,jeniemonok7@gmail.com,7103091004000200 - UMK - 12,7103091004000200,KIOS WI WIDYA JANIS,KELURAHAN TONA 1,- / 14224418,-,-,UMK,0,-,-,approved by pengawas,CAPI,jeniemonok7@gmail.comPengawas,-,UMK,(091) TAHUNA TIMUR,Djon,Tidak
428,jeniemonok7@gmail.com,7103091004000200 - UMK - 14,7103091004000200,TAYLOR CHRISTIN SURYA WATI SALENSEHE,KELURAHAN TONA 1,- / 14957370,-,-,UMK,0,-,-,approved by pengawas,CAPI,jeniemonok7@gmail.comPengawas,-,UMK,(091) TAHUNA TIMUR,Djon,Tidak
464,jeniemonok7@gmail.com,7103091004000200 - UMK - 7,7103091004000200,BENGKEL SHEREN RYAN MAYCHRISTY DAMASAR,KELURAHAN TONA 1,- / 13563764,-,-,UMK,0,-,-,approved by pengawas,CAPI,jeniemonok7@gmail.comPengawas,-,UMK,(091) TAHUNA TIMUR,Djon,Tidak
466,jeniemonok7@gmail.com,7103091004000100 - UMK - 10,7103091004000100,FARIDA SANUSI,KAMPUNG LAPANGO 1,- / 30025652,-,-,UMK,0,95815,-,approved by pengawas,CAPI,jeniemonok7@gmail.comPengawas,-,UMK,(091) TAHUNA TIMUR,Djon,Tidak
468,jeniemonok7@gmail.com,7103091004000100 - UMK - 23,7103091004000100,WISMA BUNAKEN,BELAKANG P U DAERAH,- / 97389704,-,-,UMK,0,-,-,approved by pengawas,CAPI,jeniemonok7@gmail.comPengawas,-,UMK,(091) TAHUNA TIMUR,Djon,Tidak
472,jeniemonok7@gmail.com,7103091004000100 - UMK - 9,7103091004000100,MUKSIM SALIPATI,NGALIPAENG II,- / 28968595,-,-,UMK,0,95815,-,approved by pengawas,CAPI,jeniemonok7@gmail.comPengawas,-,UMK,(091) TAHUNA TIMUR,Djon,Tidak
473,jeniemonok7@gmail.com,7103091004000100 - UMK - 11,7103091004000100,LUAS MAKAWAWA,KAMPUNG LAPANGO I,- / 30707636,-,-,UMK,0,95815,-,approved by pengawas,CAPI,jeniemonok7@gmail.comPengawas,-,UMK,(091) TAHUNA TIMUR,Djon,Tidak
474,jeniemonok7@gmail.com,7103091004000100 - UMK - 22,7103091004000100,UD CHRISAN,JLN LOPER TONA 1,- / 97389703,-,-,UMK,0,-,-,approved by pengawas,CAPI,jeniemonok7@gmail.comPengawas,-,UMK,(091) TAHUNA TIMUR,Djon,Tidak
477,jeniemonok7@gmail.com,7103091004000100 - UMK - 4,7103091004000100,KIOS SEMBAKO BUNIANI KATIANDAGHO,KELURAHAN TONA 1,- / 14846722,-,-,UMK,0,95815,-,approved by pengawas,CAPI,jeniemonok7@gmail.comPengawas,-,UMK,(091) TAHUNA TIMUR,Djon,Tidak
484,jeniemonok7@gmail.com,7103091004000100 - UMK - 21,7103091004000100,PENGUSAHA ANGKUTAN DEMINDER,JLN BARU KALUHAGI,- / 97389649,-,-,UMK,0,95815,-,approved by pengawas,CAPI,jeniemonok7@gmail.comPengawas,-,UMK,(091) TAHUNA TIMUR,Djon,Tidak


## 5. Simpan Hasil Detail (Keluarga & Usaha)
Menyimpan data hasil pemfilteran detail ke file CSV.

In [5]:
# Simpan file detail
df_kel.to_csv("tidak_ditemukan_keluarga.csv", index=False)
df_us.to_csv("tidak_ditemukan_usaha.csv", index=False)
print("File detail berhasil disimpan!")

File detail berhasil disimpan!


## 6. Membuat File Agregat per Kecamatan
Membuat file agregat per kecamatan untuk keluarga, usaha, dan gabungan (Keluarga + Usaha), serta menampilkan tabel gabungan.

In [6]:
# Buat base DataFrame berisi seluruh nama kecamatan unik dari referensi
base_kec = pd.DataFrame({'nama_kec': koseka_df['nama_kec'].unique()})

# Agregat Keluarga
agg_kel = df_kel.groupby('nama_kec').size().reset_index(name='keluarga_tidak_ditemukan')
agg_kel_full = pd.merge(base_kec, agg_kel, on='nama_kec', how='left').fillna(0)
agg_kel_full['keluarga_tidak_ditemukan'] = agg_kel_full['keluarga_tidak_ditemukan'].astype(int)
agg_kel_full.to_csv("agregat_tidak_ditemukan_keluarga.csv", index=False)

# Agregat Usaha
agg_us = df_us.groupby('nama_kec').size().reset_index(name='usaha_tidak_ditemukan')
agg_us_full = pd.merge(base_kec, agg_us, on='nama_kec', how='left').fillna(0)
agg_us_full['usaha_tidak_ditemukan'] = agg_us_full['usaha_tidak_ditemukan'].astype(int)
agg_us_full.to_csv("agregat_tidak_ditemukan_usaha.csv", index=False)

# Agregat Gabungan
agg_gab = pd.merge(agg_kel_full, agg_us_full, on='nama_kec', how='outer').fillna(0)
agg_gab['keluarga_tidak_ditemukan'] = agg_gab['keluarga_tidak_ditemukan'].astype(int)
agg_gab['usaha_tidak_ditemukan'] = agg_gab['usaha_tidak_ditemukan'].astype(int)
agg_gab['total_tidak_ditemukan'] = agg_gab['keluarga_tidak_ditemukan'] + agg_gab['usaha_tidak_ditemukan']
agg_gab = agg_gab.sort_values(by='nama_kec')
agg_gab.to_csv("agregat_tidak_ditemukan_gabungan.csv", index=False)

print("File agregat berhasil disimpan!")
print("\nTabel Agregat Gabungan:")
agg_gab

File agregat berhasil disimpan!

Tabel Agregat Gabungan:


,nama_kec,keluarga_tidak_ditemukan,usaha_tidak_ditemukan,total_tidak_ditemukan
0,(040) MANGANITU SELATAN,49,75,124
1,(041) TATOARENG,90,35,125
2,(050) TAMAKO,99,71,170
3,(060) TABUKAN SELATAN,23,104,127
4,(061) TABUKAN SELATAN TENGAH,23,21,44
5,(062) TABUKAN SELATAN TENGGARA,20,9,29
6,(070) TABUKAN TENGAH,64,50,114
7,(080) MANGANITU,50,79,129
8,(090) TAHUNA,65,156,221
9,(091) TAHUNA TIMUR,51,74,125
